In [1]:
import os, cv2, numpy as np, pandas as pd
from skimage.feature import hog, local_binary_pattern
from collections import Counter
from tqdm import tqdm

REF_DIR    = 'clean_references'
TEST_DIR   = 'test/test'
PRED_DIR   = 'test_pred'
SAMPLE_CSV = 'sample_submission.csv'
OUT_CSV    = 'submission_fused.csv'

MIN_AREA    = 10000
MORPH_KSIZE = 9

HOG_ORIENT   = 9
PIXELS_CELL  = (16,16)
CELLS_BLOCK  = (2,2)
REF_SIZE     = (128,128)

LBP_P       = 8
LBP_R       = 1
LBP_METHOD  = 'uniform'
LBP_N_BINS  = LBP_P + 2

H_BINS      = 50
V_BINS      = 50

SIM_THRESH  = 0.6

# mapping sample cols → ref filenames
col_to_ref = {
  "Jelly White":"Jelly_White",   "Jelly Milk":"Jelly_Milk",
  "Jelly Black":"Jelly_Black",   "Amandina":"Amandina",
  "Crème brulée":"Creme_brulee", "Triangolo":"Triangolo",
  "Tentation noir":"Tentation_noir","Comtesse":"Comtesse",
  "Noblesse":"Noblesse",         "Noir authentique":"Noir_authentique",
  "Passion au lait":"Passion_au_lait","Arabia":"Arabia",
  "Stracciatella":"Stracciatella",
}

In [2]:
def build_ref_descriptors(ref_dir):
    refs = {}
    for fn in sorted(os.listdir(ref_dir)):
        if not fn.lower().endswith(('.png','.jpg','.jpeg')): continue
        cls = os.path.splitext(fn)[0]
        img = cv2.imread(os.path.join(ref_dir,fn))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        # resize to REF_SIZE
        gray_r = cv2.resize(gray, REF_SIZE)
        # HOG
        hdesc = hog(gray_r, orientations=HOG_ORIENT,
                    pixels_per_cell=PIXELS_CELL, cells_per_block=CELLS_BLOCK,
                    block_norm='L2-Hys', feature_vector=True)
        # LBP
        lbp = local_binary_pattern(gray_r, LBP_P, LBP_R, LBP_METHOD)
        lbp_hist, _ = np.histogram(lbp.ravel(), bins=LBP_N_BINS, range=(0,LBP_N_BINS))
        lbp_hist = lbp_hist.astype(float); lbp_hist /= lbp_hist.sum()+1e-6
        # HSV hist
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        h_hist = cv2.calcHist([hsv],[0],None,[H_BINS],[0,180]).flatten()
        v_hist = cv2.calcHist([hsv],[2],None,[V_BINS],[0,256]).flatten()
        h_hist /= h_hist.sum()+1e-6; v_hist /= v_hist.sum()+1e-6
        # concat & L2 normalize
        desc = np.hstack([hdesc, lbp_hist, h_hist, v_hist])
        desc /= np.linalg.norm(desc)+1e-6
        refs[cls] = desc
        print(f"Ref '{cls}': descriptor length {len(desc)}")
    return refs

ref_descs = build_ref_descriptors(REF_DIR)
classes   = list(ref_descs.keys())

FileNotFoundError: [Errno 2] No such file or directory: 'clean_references'

In [ ]:
def segment_pieces(img):
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    border = np.hstack([gray[0],gray[-1],gray[:,0],gray[:,-1]])
    bg = int(np.median(border))
    diff = cv2.absdiff(gray, np.full_like(gray,bg))
    _,mask = cv2.threshold(diff,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(MORPH_KSIZE,MORPH_KSIZE))
    mask = cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kern,iterations=3)
    mask = cv2.dilate(mask,kern,iterations=2)
    cnts,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    pieces=[]
    for c in cnts:
        if cv2.contourArea(c)<MIN_AREA: continue
        x,y,w,h = cv2.boundingRect(c)
        m = np.zeros_like(gray); cv2.drawContours(m,[c],-1,255,-1)
        pieces.append((m,(x,y,w,h)))
    return pieces

def classify_and_draw(img, ref_descs):
    pieces = segment_pieces(img)
    cnts   = Counter()
    vis    = img.copy()
    for mask,(x,y,w,h) in pieces:
        patch = img[y:y+h,x:x+w]
        gray = cv2.cvtColor(patch,cv2.COLOR_BGR2GRAY)
        gray_r = cv2.resize(gray, REF_SIZE)
        # HOG+LBP
        hdesc = hog(gray_r, orientations=HOG_ORIENT,
                    pixels_per_cell=PIXELS_CELL, cells_per_block=CELLS_BLOCK,
                    block_norm='L2-Hys', feature_vector=True)
        lbp = local_binary_pattern(gray_r, LBP_P, LBP_R, LBP_METHOD)
        lbp_hist,_ = np.histogram(lbp.ravel(), bins=LBP_N_BINS, range=(0,LBP_N_BINS))
        lbp_hist = lbp_hist.astype(float); lbp_hist/=lbp_hist.sum()+1e-6
        # HSV
        hsv = cv2.cvtColor(patch,cv2.COLOR_BGR2HSV)
        h_hist = cv2.calcHist([hsv],[0],None,[H_BINS],[0,180]).flatten()
        v_hist = cv2.calcHist([hsv],[2],None,[V_BINS],[0,256]).flatten()
        h_hist/=h_hist.sum()+1e-6; v_hist/=v_hist.sum()+1e-6
        desc = np.hstack([hdesc, lbp_hist, h_hist, v_hist])
        desc/=np.linalg.norm(desc)+1e-6

        # nearest neighbor
        best_cls, best_sim = None, -1
        for cls, rdesc in ref_descs.items():
            sim = float(np.dot(desc, rdesc))
            if sim>best_sim:
                best_sim, best_cls = sim, cls

        if best_sim>=SIM_THRESH:
            cnts[best_cls]+=1
            cv2.rectangle(vis,(x,y),(x+w,y+h),(0,255,0),2)
            cv2.putText(vis,f"{best_cls}:{best_sim:.2f}",(x,y-5),
                        cv2.FONT_HERSHEY_SIMPLEX,3,(0,255,0),2)

    # zero-fill
    for c in classes:
        cnts.setdefault(c,0)
    return dict(cnts), vis

os.makedirs(PRED_DIR,exist_ok=True)
sample = pd.read_csv(SAMPLE_CSV)
out    = sample.copy()

for idx,row in tqdm(sample.iterrows(),total=len(sample)):
    img_id = str(row['id'])
    fn     = f"L{img_id}.jpg"
    path   = os.path.join(TEST_DIR,fn)
    if not os.path.isfile(path):
        path = os.path.join(TEST_DIR,f"L{img_id}.JPG")
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)

    cnts, vis = classify_and_draw(img, ref_descs)
    # fill out
    for sub_col, ref_key in col_to_ref.items():
        out.at[idx, sub_col] = int(cnts.get(ref_key,0))

    cv2.imwrite(os.path.join(PRED_DIR,fn), vis)

out.to_csv(OUT_CSV,index=False)